# 4.0.1-Find Highly Variable Features

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressMessages(suppressWarnings({
library(Seurat)
library(Signac)
library(tidyverse)
library(scater) 
library(scran)
library(here)
library(glue)
}))

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))
set.seed(0)

## Load Data

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.0.0-combined_object_unify.rds")))

In [ ]:
combined_obj <- combined_obj %>% NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 

### High variable features

Remove dimorfic genes according to [this paper](https://www.nature.com/articles/ncomms10717/figures/1).

In [ ]:
genes_dmf <- c("XACT", "HTR2C",
               "XIST", "DDX3Y","USP9Y","NLGN4Y", "KDM5D", "EIF1AY", "UBA1")
genes_out <- c(genes_dmf, top_c) 

In [ ]:
hvgs_options <- list()

### Option A 

Infer HVG again from combined object but splitting by patient

In [ ]:
list_objects <- SplitObject(combined_obj, split.by = "patient")
donor_ids <- names(list_objects)

In [ ]:
find_hvgs <-  function(obj) {
    
    # remove genes from matrix before modulating variance and mean
    DefaultAssay(obj) <- "RNA"
    df_meta <- obj@meta.data
    counts <- obj[['RNA']]@counts
    counts <- counts[-(which(rownames(counts) %in% genes_out)),]
    obj2 <- CreateSeuratObject(counts = counts, meta.data = df_meta)
    
    # normalize again
    obj2 <- obj2 %>% 
            NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 
    
    # compute the variance and mean of the log-expression values 
    sce <- as.SingleCellExperiment(obj2)
    gene_var <- modelGeneVar(sce)
    
    tops <- gene_var %>% 
        as.data.frame() %>% 
        arrange(desc(total)) %>% 
        head(n=20)

    plt <- gene_var %>% 
      as.data.frame() %>% 
      ggplot(aes(mean, total)) +
      geom_point() +
      ggtitle(unique(df_meta$patient)) +
      geom_line(aes(y = tech), colour = "dodgerblue", size = 1) +
      labs(x = "Mean of log-expression", y = "Variance of log-expression")+
      geom_text(data=tops, aes(mean,total,label=rownames(tops))) +
      theme(plot.title = element_text(hjust = 0.5))

    hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)
    return(list(plot = plt, hvgs_pat = hvgs))
}

In [ ]:
modelling <- lapply(list_objects, find_hvgs)

In [ ]:
list_plt <- lapply(modelling, function(el) {
    el$plot
})

In [ ]:
length(list_plt)

In [ ]:
options(repr.plot.width = 16, repr.plot.height = 22)
cp <- cowplot::plot_grid(plotlist = list_plt,
                   align = "hv",
                   axis = "trbl",
                   ncol = 4,
                   nrow = 5)
cp

In [ ]:
list_hvg <- lapply(modelling, function(el) {
    el$hvgs_pat
})

In [ ]:
length(list_hvg)

In [ ]:
hvg_common <- find_perc(list_hvg, perc = 3/16)
length(hvg_common)

In [ ]:
hvgs_options[["optionA"]] <- as.vector(hvg_common)

#### Highly variable peaks 

Find top peaks per patient in from the consensus unified peak calls and use a cut off of 10% which means 
to include features with >10 total counts in the set of VariableFeatures

In [ ]:
list_hvp <- lapply(list_objects, function(obj) {
  DefaultAssay(obj) <- "ATAC_unified"
  hvp <-VariableFeatures(FindTopFeatures(obj[["ATAC_unified"]], min.cutoff = 10))
  return(hvp)
})

In [ ]:
hvp_common <- Reduce(intersect, list_hvp)
length(hvp_common)

In [ ]:
hvgs_options[["ATAC"]] <- as.vector(hvp_common)

### Option B

Use the HVG obtained during the individual processing of each sample

In [ ]:
list_objects <- readRDS(here(glue("{int}/{robj_dir}/4.0.0-list_objects_unify.rds")))
patient_ids <- names(list_objects)

In [ ]:
list_objects$p1

In [ ]:
list_hvg <- lapply(list_objects, function(obj) {
        VariableFeatures(obj)
})

In [ ]:
hvg_common <- find_perc(list_hvg,perc = 3/16)
length(hvg_common)       

In [ ]:
hvgs_options[["optionB"]] <- hvg_common

### Option C

Remove batch effect by finding HVG unique to the batch

In [ ]:
list_objects <- SplitObject(combined_obj, split.by = "subproj")

In [ ]:
find_hvgs <-  function(obj) {
    
    batch <- unique(obj$subproj)
    
    # remove genes from matrix before modulating variance and mean
    DefaultAssay(obj) <- "RNA"
    df_meta <- obj@meta.data
    counts <- obj[['RNA']]@counts
    counts <- counts[-(which(rownames(counts) %in% genes_out)),]
    obj2 <- CreateSeuratObject(counts = counts, meta.data = df_meta)
    
    # normalize again
    obj2 <- obj2 %>% 
            NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 
    
    # compute the variance and mean of the log-expression values 
    sce <- as.SingleCellExperiment(obj2)
    gene_var <- modelGeneVar(sce)
    
    tops <- gene_var %>% 
        as.data.frame() %>% 
        arrange(desc(total)) %>% 
        head(n=20)

    plt <- gene_var %>% 
      as.data.frame() %>% 
      ggplot(aes(mean, total)) +
      geom_point() +
      geom_line(aes(y = tech), colour = "dodgerblue", size = 1) +
      labs(x = "Mean of log-expression", y = "Variance of log-expression")+
      geom_text(data=tops, aes(mean,total,label=rownames(tops))) + 
      ggtitle(batch) + 
      theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

    hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)
    return(list(plot = plt, hvgs_pat = hvgs))
}

In [ ]:
modelling <- lapply(list_objects, find_hvgs)

In [ ]:
list_plots <- lapply(modelling, function(el) {
    el$plot
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
list_hvg <- lapply(modelling, function(el) {
    hvg <- el$hvgs_pat
    print(length(hvg))
    hvg
})

In [ ]:
hvg_common <- find_perc(list_hvg, perc = 2/7)
length(hvg_common)

In [ ]:
hvgs_options[["optionC"]] <- hvg_common

## Save

In [ ]:
saveRDS(hvgs_options, here(glue("{int}/{robj_dir}/4.0.1-hvf_list.rds")))

## Session

In [ ]:
sessionInfo()